# Simulation of a silicon npn bipolar transistor

The graphs of the carrier concentrations and the output characteristics are stored in the folder fig.

In [ ]:
import os

import numpy as np

import matplotlib.pyplot as plt

from synumses.one_dimension.scharfetter_gummel_bernoulli import bernoulli, jacobian, update_b, first_jacobian, first_update_b, hole_current_density, electron_current_density

import synumses.one_dimension.parameters as parameters

from synumses.one_dimension.functions import  ohm_potential, calc_p_density, calc_n_density, calc_recombination

import synumses.one_dimension.solver as solver

## Definition of the transistor

Defining the grid points *parameters.n* and the length *parameters.lx* and the doping levels.

In [ ]:
parameters.n = 600
parameters.lx = 1200E-9

parameters.init_geometry()


#parameters.Cau = 0*1E-24  
# Definition of the doping levels
Nd_Emitter   = 1.2E24
Na_Base      = 4.0E22
Nd_Collector = 2.0E22


Eg = 1.12

# np transition at x1
# pn transition at x2
x1 = 400E-9
x2 = 800E-9

for i in range(0, parameters.n):
    
    parameters.Ec[i] = Eg
    parameters.Ev[i] = 0
    
    if parameters.pos_x[i] < x1:
        
        parameters.C[i] = + Nd_Emitter
    elif parameters.pos_x[i] < x2: 
        parameters.C[i] = - Na_Base
    else:
        parameters.C[i] = + Nd_Collector

## Plotting the parameters of the transistor
Now, we plot the default band diagramm and the doping levels.

In [ ]:
#
# Plot band diagramm and doping
#
fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Conduction and valence band", fontsize=16)

axis.plot(parameters.pos_x * 1E9, parameters.Ec, label='Conduction band')
axis.plot(parameters.pos_x * 1E9, parameters.Ev, label='Valence band')

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$E$ in $\mathrm{eV}$")

axis.legend()
plt.show()

# Doping level
fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Doping", fontsize=16)

#axis.semilogy(np.abs(parameters.C), label='Doping')
axis.plot(parameters.pos_x * 1E9, np.abs(parameters.C), label='Doping')

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$N_\mathrm{A} / N_\mathrm{D}$ in $\mathrm{m^{-3}}$")

axis.legend()
plt.show()


## Solve without biasing

Now, let's plot the band diagramm considering diffusion but no biasing.<br>
This must be done before simulating with bias.

In [ ]:
parameters.init_potential()

solver.solve_no_bias()

fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Potential from doping and considering diffusion", fontsize=16)

axis.plot(parameters.pos_x * 1E9, parameters.u[0::3])

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$\varphi$ in $\mathrm{V}$")

axis.grid()
plt.show()

## First ramping $U_\mathrm{BE}$

Now, let's ramp from $U_\mathrm{BE} = 0\,\mathrm{V}$ to $U_\mathrm{BE} = 0.6\,\mathrm{V}$.

The function *solver.solve_bias_center_boundary(.0 , .0, u_start, 650E-9, "p")* sets a further boundary condition at
the location $650\,\mathrm{nm}$ to *u_start*. As the base is p-doped the potential and the Quasi-Fermi-Level for the holes is set by
the boundary voltage. The Quasi-Fermi-Level for the electrons is not set and is calculated.


In [ ]:
solver.solve_no_bias()

u_start = .00
u_stop  = .6
u_step  = 0.1


while (u_start <= u_stop):
    print("Spannung", u_start)
    solver.solve_bias_center_boundary(.0 , .0, u_start, 650E-9, "p")
    u_start += u_step
    

    
    
# Potential
fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Potential", fontsize=16)

axis.plot(parameters.pos_x * 1E9, parameters.u[0::3], label='Potential')

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$\varphi$ in $\mathrm{V}$")

#plt.ylim((0.95,0.96))
axis.legend()
plt.show()

# Fermi-Level
fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Fermi Level", fontsize=16)

axis.plot(parameters.pos_x * 1E9, parameters.u[1::3], label='Fermi Level p')
axis.plot(parameters.pos_x * 1E9, parameters.u[2::3], label='Fermi Level n')

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$\varphi$ in $\mathrm{V}$")

axis.legend()
plt.show() 
    
    
# Hole and electron current density 
j_p =     hole_current_density()
j_n =     electron_current_density()

fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Current density of holes and electrons", fontsize=16)

axis.plot(parameters.pos_x * 1E9, j_p, label='Hole current density')
axis.plot(parameters.pos_x * 1E9, j_n, label='Electron current density')
axis.plot(parameters.pos_x * 1E9, j_n+j_p, label='Total current density')

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$j_\mathrm{p}, j_\mathrm{n}, j_\mathrm{total}$ in $\mathrm{\dfrac{A}{m}}$")

axis.legend()
axis.grid()
plt.show()
 

## Then ramp $U_\mathrm{CE}$

Now, let's ramp $U_\mathrm{CE}$ to $3\,\mathrm{V}$.

Save the carrier concentration in the folder *fig*.

And store the *base_current*, the *collector_current*, and the *collector_voltage*.

In [ ]:
base_current = []
collector_current = []
collector_voltage = []

u_base = 0.6

u_start = 0.0
u_stop  = 3.0
u_step  = 0.05

try:
    os.makedirs("./fig")
except FileExistsError:
    # directory already exists
    pass

while (u_start <= u_stop):
    print("Spannung", u_start)
    solver.solve_bias_center_boundary(.0 , u_start, u_base, 650E-9, "p")
    
    j_p =     hole_current_density()
    j_n =     electron_current_density()
    
    base_current.append(-j_p[0])
    collector_current.append(-j_n[0])
    collector_voltage.append(u_start)
    
    u_start += u_step
    
    # Hole and electron density
    calc_p_density()
    calc_n_density()

    p = parameters.p_density
    n = parameters.n_density

    fig, axis = plt.subplots(1,1, sharey=True)

    fig.suptitle("Density of holes and electrons", fontsize=16)

    axis.plot(parameters.pos_x * 1E9, p, label='Hole density')
    axis.plot(parameters.pos_x * 1E9, n, label='Electron density')

    plt.ylim((0,.4E20))

    axis.set_xlabel(r"x in $\mathrm{nm}$")
    axis.set_ylabel(r"$j_n$ in $\mathrm{\dfrac{A}{m^2}}$")

    axis.grid()
    axis.legend(loc = "upper left")
    figname = './fig/carrier_concentration_{:0>4}.png'.format(str(round((u_start*1000.0))))
    plt.savefig(figname)
    plt.close()
    

## Distribution of the electrons and holes

Distribution of the electrons and holes along the semiconductor.

In [ ]:
#
# Plot hole and electron density
#
calc_p_density()
calc_n_density()

p = parameters.p_density
n = parameters.n_density

fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Density of holes and electrons", fontsize=16)

axis.plot(parameters.pos_x * 1E9, p, label='Hole density')
axis.plot(parameters.pos_x * 1E9, n, label='Electron density')

plt.ylim((0,.4E20))

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$p, n$ in $\mathrm{\dfrac{1}{m^3}}$")

axis.grid()
axis.legend()
plt.show()




# Plot the output characteristics

In [ ]:
# Collector current density 

fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Output characteristsics", fontsize=16)

axis.plot(collector_voltage,collector_current, label='Collector current density')

#plt.ylim((0,.2E20))

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$j_\mathrm{c}$ in $\mathrm{\dfrac{A}{m^2}}$")

axis.legend()
plt.show()

In [ ]:
# Potential
fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Potential", fontsize=16)

axis.plot(parameters.pos_x * 1E9, parameters.u[0::3], label='Potential')

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$\varphi$ in $\mathrm{V}$")

#plt.ylim((0.95,0.96))
axis.legend()
plt.show()

# Fermi-Level
fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Fermi Level", fontsize=16)

axis.plot(parameters.pos_x * 1E9, parameters.u[1::3], label='Fermi Level p')
axis.plot(parameters.pos_x * 1E9, parameters.u[2::3], label='Fermi Level n')

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$\varphi$ in $\mathrm{V}$")

axis.legend()
plt.show() 
    
    
# Hole and electron current density 
j_p =     hole_current_density()
j_n =     electron_current_density()

fig, axis = plt.subplots(1,1, sharey=True)
fig.suptitle("Current density of holes and electrons", fontsize=16)

axis.plot(parameters.pos_x * 1E9, j_p, label='Hole current density')
axis.plot(parameters.pos_x * 1E9, j_n, label='Electron current density')
axis.plot(parameters.pos_x * 1E9, j_n+j_p, label='Total current density')

axis.set_xlabel(r"x in $\mathrm{nm}$")
axis.set_ylabel(r"$j_\mathrm{p}, j_\mathrm{n}, j_\mathrm{total}$ in $\mathrm{\dfrac{A}{m^2}}$")

axis.legend()
axis.grid()
plt.show()

Save the output characteristsics with bias point marked.

In [ ]:
for bias in collector_voltage:
    fig, axis = plt.subplots(1,1, sharey=True)
    fig.suptitle("Output characteristsics", fontsize=16)

    idx = np.abs(np.subtract(collector_voltage, bias)).argmin()
    print(idx)
    axis.plot(collector_voltage,
              collector_current,
              '-rD', markevery=[idx],
              label='Hole density')

    #plt.ylim((0,.2E20))

    axis.set_xlabel(r"x in $\mathrm{nm}$")
    axis.set_ylabel(r"$j_\mathrm{c}$ in $\mathrm{\dfrac{A}{m^2}}$")

    
    axis.grid()
    axis.legend()
    
    figname = './fig/output_characteristsics_{:0>4}.png'.format(str(round((bias*1000.0))))
    plt.savefig(figname)
    plt.close()

# Animation

Now let's make an animation to see the hole and electron densities depending on the biasing.<br>
It's stored in the file Animation.gif.<br>
The saturation and Early effect can be seen.

In [ ]:
from PIL import Image, ImageDraw

frames = []

for bias in collector_voltage[1:]:
    figname1 = './fig/output_characteristsics_{:0>4}.png'.format(str(round((bias*1000.0))))
    figname2 = './fig/carrier_concentration_{:0>4}.png'.format(str(round((bias*1000.0))))
    gr1 = Image.open(figname1)
    gr2 = Image.open(figname2)
    
    dst = Image.new('RGB', (gr1.width, gr1.height + gr2.height))
    dst.paste(gr1, (0, 0))
    dst.paste(gr2, (0, gr1.height))
    
    frames.append(dst)          
    
frames[0].save('Animation.gif', format='GIF',
    append_images=frames[1:], save_all=True, duration=100, loop=0)
